# Encode latents for diffusion training


| Cell | When to use | Input CSV needs |
|------|-------------|------------------|
| **3 — Normal** | Organism-only training (no potency levels) | `sequence`/`peptide` + `organism` |
| **4 — Leveled** | Training with Weak/Moderate/Strong levels | above + `potency_level` |


### Project paths

This notebook lives in `notebooks/`. Shared code is under `src/`, data under `data/`.
On Colab, set `ROOT` to your cloned repo (or Drive copy) instead of `Path('..')`.


In [ ]:
import sys
from pathlib import Path

# Repo root (parent of notebooks/). On Colab, replace with your clone path, e.g.:
# ROOT = Path('/content/drive/MyDrive/your-thesis-repo')
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.paths import DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, PROJECT_ROOT
print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('CHECKPOINTS  :', CHECKPOINTS_DIR)
print('RESULTS      :', RESULTS_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys

INPUT_PATH = "/content/drive/MyDrive/DBAASP_Bioinformatics"
os.chdir(INPUT_PATH)
sys.path.append(INPUT_PATH)

print("Working dir:", os.getcwd())

In [ ]:
VAE_WEIGHTS = INPUT_PATH + "/vae_weights.pth"
OUT_DIR       = INPUT_PATH + "/latent_withLevelings/2org-baumannii-enterica/cond_latents_new"
BATCH_SIZE    = 512

CSV_NORMAL = INPUT_PATH + "/databases/enterica_baumannii_levels.csv"

CSV_LEVELED = INPUT_PATH + "/databases/enterica_baumannii_levels.csv"

print("Normal CSV :", CSV_NORMAL)
print("Leveled CSV:", CSV_LEVELED)
print("Output dir :", OUT_DIR)

## Initial encoder (sequence + organism only, no potency levels)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from src.models.transvae import create_VAE, vae_data_gen, w2i, params
from src.utils import pick_col, filter_aa_pairs

def _encode_vae(model, sequences, batch_size):
    data = np.array([[s] for s in sequences])
    encoded = vae_data_gen(data, params["src_len"], char_dict=w2i)
    loader = DataLoader(encoded, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True, drop_last=False)
    model.eval()
    chunks = []
    with torch.no_grad():
        for batch in loader:
            mols = batch[:, :-1].long().cuda()
            mask = (mols != w2i["_"]).unsqueeze(-2).cuda()
            _, mu, _, _ = model.encode(mols, mask)
            chunks.append(mu.cpu().numpy())
    return np.concatenate(chunks, axis=0).astype(np.float32)


def run_normal_encoder(csv_path, out_dir, vae_weights, batch_size):
    df = pd.read_csv(csv_path)
    if df.shape[1] == 1:
        df = pd.read_csv(csv_path, sep="\t")
    print(f"[normal] Loaded {len(df)} rows: {list(df.columns)}")

    seq_col = pick_col(df, ["sequence", "peptide", "Sequence"], "sequence")
    org_col = pick_col(df, ["organism", "Organism"], "organism")

    sequences = df[seq_col].astype(str).str.upper().str.strip().tolist()
    organisms = df[org_col].astype(str).str.strip().tolist()
    sequences, organisms, _, dropped = filter_aa_pairs(sequences, organisms)
    if dropped:
        print(f"Dropped {dropped} sequences with non-standard amino acids")
    print(f"[normal] Encoding {len(sequences)} sequences")

    unique_orgs = sorted(set(organisms))
    org_to_id = {o: i for i, o in enumerate(unique_orgs)}
    organism_ids = np.array([org_to_id[o] for o in organisms], dtype=np.int64)

    for i, org in enumerate(unique_orgs):
        print(f"  [{i}] {org} — {(organism_ids == i).sum()}")

    os.makedirs(out_dir, exist_ok=True)
    np.save(f"{out_dir}/organism_map.npy", np.array(unique_orgs))
    np.save(f"{out_dir}/cond_organisms.npy", organism_ids)

    model = create_VAE()
    model.load_state_dict(torch.load(vae_weights, map_location="cpu"))
    model.cuda().eval()
    for p in model.parameters():
        p.requires_grad_(False)

    latents = _encode_vae(model, sequences, batch_size)
    np.save(f"{out_dir}/cond_latents.npy", latents)

    print(f"\n[normal] Saved to {out_dir}/")
    print(f"  cond_latents.npy    {latents.shape}")
    print(f"  cond_organisms.npy  {organism_ids.shape}")
    print(f"  organism_map.npy    ({len(unique_orgs)},)")
    print("  (no cond_levels.npy — organism-only mode)")
    return latents.shape


run_normal_encoder(CSV_NORMAL, OUT_DIR, VAE_WEIGHTS, BATCH_SIZE)


## Leveled encoder (organism + Weak/Moderate/Strong)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from src.models.transvae import create_VAE, vae_data_gen, w2i, params
from src.utils import pick_col, filter_aa_pairs, parse_level_column, LEVEL_NAMES

def _encode_vae(model, sequences, batch_size):
    data = np.array([[s] for s in sequences])
    encoded = vae_data_gen(data, params["src_len"], char_dict=w2i)
    loader = DataLoader(encoded, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True, drop_last=False)
    model.eval()
    chunks = []
    with torch.no_grad():
        for batch in loader:
            mols = batch[:, :-1].long().cuda()
            mask = (mols != w2i["_"]).unsqueeze(-2).cuda()
            _, mu, _, _ = model.encode(mols, mask)
            chunks.append(mu.cpu().numpy())
    return np.concatenate(chunks, axis=0).astype(np.float32)


def run_leveled_encoder(csv_path, out_dir, vae_weights, batch_size):
    df = pd.read_csv(csv_path)
    if df.shape[1] == 1:
        df = pd.read_csv(csv_path, sep="\t")
    print(f"[leveled] Loaded {len(df)} rows: {list(df.columns)}")

    seq_col = pick_col(df, ["sequence", "peptide", "Sequence"], "sequence")
    org_col = pick_col(df, ["organism", "Organism"], "organism")
    level_ids, level_col = parse_level_column(df)
    print(f"[leveled] Using potency column: '{level_col}'")

    sequences = df[seq_col].astype(str).str.upper().str.strip().tolist()
    organisms = df[org_col].astype(str).str.strip().tolist()
    sequences, organisms, level_ids, dropped = filter_aa_pairs(sequences, organisms, level_ids.tolist())
    if dropped:
        print(f"Dropped {dropped} sequences with non-standard amino acids")
    level_ids = np.array(level_ids, dtype=np.int64)
    print(f"[leveled] Encoding {len(sequences)} sequences")

    unique_orgs = sorted(set(organisms))
    org_to_id = {o: i for i, o in enumerate(unique_orgs)}
    organism_ids = np.array([org_to_id[o] for o in organisms], dtype=np.int64)

    print("\nOrganisms:")
    for i, org in enumerate(unique_orgs):
        print(f"  [{i}] {org} — {(organism_ids == i).sum()}")
    print("\nLevels:")
    for i, name in enumerate(LEVEL_NAMES):
        print(f"  [{i}] {name} — {(level_ids == i).sum()}")

    os.makedirs(out_dir, exist_ok=True)
    np.save(f"{out_dir}/organism_map.npy", np.array(unique_orgs))
    np.save(f"{out_dir}/cond_organisms.npy", organism_ids)
    np.save(f"{out_dir}/level_map.npy", np.array(LEVEL_NAMES))
    np.save(f"{out_dir}/cond_levels.npy", level_ids)

    model = create_VAE()
    model.load_state_dict(torch.load(vae_weights, map_location="cpu"))
    model.cuda().eval()
    for p in model.parameters():
        p.requires_grad_(False)

    latents = _encode_vae(model, sequences, batch_size)
    np.save(f"{out_dir}/cond_latents.npy", latents)

    print(f"\n[leveled] Saved to {out_dir}/")
    print(f"  cond_latents.npy    {latents.shape}")
    print(f"  cond_organisms.npy  {organism_ids.shape}")
    print(f"  cond_levels.npy     {level_ids.shape}")
    print(f"  level_map.npy       {LEVEL_NAMES}")
    print(f"  organism_map.npy    ({len(unique_orgs)},)")
    return latents.shape


run_leveled_encoder(CSV_LEVELED, OUT_DIR, VAE_WEIGHTS, BATCH_SIZE)